In [ ]:
import numpy as np
#!pip install tslearn
from tslearn.datasets import UCR_UEA_datasets
import torch
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
#!pip install uni2ts.model.moirai
from uni2ts.model.moirai import MoiraiModule

ERROR: Could not find a version that satisfies the requirement uni2ts.model.moirai (from versions: none)
ERROR: No matching distribution found for uni2ts.model.moirai


ModuleNotFoundError: No module named 'uni2ts'

# Data

In [ ]:
ds = UCR_UEA_datasets()
X_train, y_train, X_test, y_test = ds.load_dataset("LSST")

n_classes = len(set(y_train))
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

# Define MoiraiEncoder

In [ ]:
MODEL_PARAMS = {
    "patch_size": 16,  # 8, 16, 32, 64, 128
    "num_samples": 100,
    "target_dim": 1,
    "feat_dynamic_real_dim": 0,
    "past_feat_dynamic_real_dim": 0,
}
SIZE = "small"  # model size: choose from {'small', 'base', 'large'}
device = "cuda"

In [ ]:
from encoder import MoiraiEncoder

encoder_ = MoiraiEncoder(
    module=MoiraiModule.from_pretrained(f"Salesforce/moirai-1.1-R-{SIZE}"),
    prediction_length=0,
    context_length=36,
    **MODEL_PARAMS,
)

In [ ]:
patches_list = [8, 16, 32, 64, 128]
train_window1 = 12 * 365 # days
prediction_window1 = 90
mape_list = []
time_list = []

train1 = df.iloc[:train_window1, :].copy()
test1 = df.iloc[train_window1:, :].copy()

for patch in patches_list:
    start_time = time.time()
    # preprocess the data
    target_tensor, is_target_observed, is_target_padded= preprocess_data(target=train1[target_colname].values)

    MODEL_PARAMS["patch_size"] = patch
    # Define the model
    model = MoiraiMoEForecast(
        module=MoiraiMoEModule.from_pretrained(f"Salesforce/moirai-moe-1.0-R-small"),
        prediction_length= prediction_window1,
        context_length=train_window1,
        **MODEL_PARAMS,
    )

    # Make the forecast
    forecast = model(
        past_target=target_tensor,
        past_observed_target=is_target_observed,
        past_is_pad=is_target_padded,
    )

    # Collect the median forecast and CI into a DataFrame
    predictions_df = get_predictions(
        forecast_tensor=forecast[0],
        train_set=train1,
        date_colname=date_colname
    )

    # Merge predictions with original data
    true_and_preds = pd.merge(test1, predictions_df, left_index=True, right_index=True, how="right")
    all_data = pd.concat([train1.iloc[-200:, :], true_and_preds])

    true_and_preds.dropna(inplace=True)
    mape = mean_absolute_percentage_error(
        y_true=true_and_preds[target_colname].values,
        y_pred=true_and_preds["median_forecast"].values
    )
    end_time = time.time()
    elapsed_time = end_time - start_time
    mape_list.append(mape)
    time_list.append(elapsed_time)

mape_batch_size_df = pd.DataFrame(
    {"patch_size": patches_list, "mape": mape_list, "elapsed_time": time_list}
)
mape_batch_size_df

# Prepare data

Preprocess data

In [ ]:
def preprocess_data(
    data: np.ndarray,
    *,
    device: str | torch.device = "cpu",
    dtype: torch.dtype = torch.float32,
):
    """
    data: np.ndarray of shape (N, T, V) = (n_individual, time, variate)
    Assumes NO missing values and NO padding in the raw data.
    """

    if not isinstance(data, np.ndarray):
        raise TypeError("data must be a numpy.ndarray")
    if data.ndim != 3:
        raise ValueError(f"Expected shape (N,T,V), got {data.shape}")

    N, T, V = data.shape

    # (N,T,V)
    past_target = torch.as_tensor(data, dtype=dtype, device=device)

    # observed mask: (N,T,V) all True no missing values
    past_observed_target = torch.ones((N, T, V), dtype=torch.bool, device=device)

    # padding mask: (N,T) all if no padding
    past_is_pad = torch.zeros((N, T), dtype=torch.bool, device=device)

    return past_target, past_observed_target, past_is_pad

In [ ]:
X_train_target_tensor, X_train_is_target_observed, X_train_is_target_padded = (
    preprocess_data(X_train, device=device)
)

X_test_target_tensor, X_test_is_target_observed, X_test_is_target_padded = (
    preprocess_data(X_test, device=device)
)


print(X_train_target_tensor.shape)
print(X_train_is_target_observed.shape)
print(X_train_is_target_padded.shape)

# Time seires encoding

In [ ]:
encoder.eval()
encoder.to(device)

with torch.no_grad():
    Z_train = encoder(
        past_target=X_train_target_tensor,
        past_observed_target=X_train_is_target_observed,
        past_is_pad=X_train_is_target_padded,
    )  # (N_train, combine_seq, d_model)

    Z_test = encoder(
        past_target=X_test_target_tensor,
        past_observed_target=X_test_is_target_observed,
        past_is_pad=X_test_is_target_padded,
    )  # (N_test, combine_seq, d_model)

Z_train.shape, Z_test.shape

In [ ]:
patches_list = [8, 16, 32, 64, 128]
train_window1 = 12 * 365 # days
prediction_window1 = 90
mape_list = []
time_list = []

train1 = df.iloc[:train_window1, :].copy()
test1 = df.iloc[train_window1:, :].copy()

for patch in patches_list:
    start_time = time.time()
    # preprocess the data
    target_tensor, is_target_observed, is_target_padded= preprocess_data(target=train1[target_colname].values)

    MODEL_PARAMS["patch_size"] = patch
    # Define the model
    model = MoiraiMoEForecast(
        module=MoiraiMoEModule.from_pretrained(f"Salesforce/moirai-moe-1.0-R-small"),
        prediction_length= prediction_window1,
        context_length=train_window1,
        **MODEL_PARAMS,
    )

    # Make the forecast
    forecast = model(
        past_target=target_tensor,
        past_observed_target=is_target_observed,
        past_is_pad=is_target_padded,
    )

    # Collect the median forecast and CI into a DataFrame
    predictions_df = get_predictions(
        forecast_tensor=forecast[0],
        train_set=train1,
        date_colname=date_colname
    )

    # Merge predictions with original data
    true_and_preds = pd.merge(test1, predictions_df, left_index=True, right_index=True, how="right")
    all_data = pd.concat([train1.iloc[-200:, :], true_and_preds])

    true_and_preds.dropna(inplace=True)
    mape = mean_absolute_percentage_error(
        y_true=true_and_preds[target_colname].values,
        y_pred=true_and_preds["median_forecast"].values
    )
    end_time = time.time()
    elapsed_time = end_time - start_time
    mape_list.append(mape)
    time_list.append(elapsed_time)

mape_batch_size_df = pd.DataFrame(
    {"patch_size": patches_list, "mape": mape_list, "elapsed_time": time_list}
)
mape_batch_size_df